In [ ]:
# %% [markdown]
# # Object Detection from Video using YOLOv12
# This notebook allows you to upload videos and detect objects using the YOLOv12 model.

# %% [markdown]
# ## 1. Installation

# %%
# Install required packages
%pip install ultralytics>=8.3.0
%pip install opencv-python
%pip install ipywidgets

# %% [markdown]
# ## 2. Import Libraries

# %%
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display, HTML, Video, clear_output
import ipywidgets as widgets
import os
import time
from pathlib import Path

# %% [markdown]
# ## 3. Load YOLOv12 Model

# %%
# Available YOLOv12 models: yolo12n, yolo12s, yolo12m, yolo12l, yolo12x
# n=nano, s=small, m=medium, l=large, x=extra large

model_options = {
    
    'yolo12l': 'Large (Slower, More Accurate)',
    'yolo12x': 'Extra Large (Slowest, Most Accurate)'
}

print("Available YOLOv12 Models:")
for model_name, description in model_options.items():
    print(f"  - {model_name}: {description}")

# %%
# Select and load model
SELECTED_MODEL = 'yolo12x'  # Change this to your preferred model

print(f"\nLoading {SELECTED_MODEL} model...")
model = YOLO(f'{SELECTED_MODEL}.pt')
print(f"✅ Model {SELECTED_MODEL} loaded successfully!")
print(f"Model classes: {len(model.names)} classes")

# %% [markdown]
# ## 4. Video Upload Functions

# %%
# For Local Jupyter - File path input
def get_video_path_local():
    """Get video path for local Jupyter notebook"""
    video_path = input("Enter the path to your video file: ")
    if os.path.exists(video_path):
        print(f"✅ Video found: {video_path}")
        return video_path
    else:
        print(f"❌ Video not found: {video_path}")
        return None

# %% [markdown]
# ## 5. Object Detection Function

# %%
def detect_objects_in_video(
    video_path: str,
    model=None,
    output_path: str = "output_detected.mp4",
    confidence_threshold: float = 0.25,
    show_progress: bool = True,
    save_frames: bool = False,
    frames_dir: str = "detected_frames"
):
    """
    Detect objects in a video using YOLOv12
    
    Parameters:
    -----------
    video_path : str
        Path to the input video
    output_path : str
        Path for the output video with detections
    confidence_threshold : float
        Minimum confidence for detections (0-1)
    show_progress : bool
        Show progress bar
    save_frames : bool
        Save individual frames with detections
    frames_dir : str
        Directory to save frames
    
    Returns:
    --------
    dict : Detection statistics
    """
    
    yolo_model = model if model is not None else globals()['model']
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"❌ Error: Cannot open video {video_path}")
        return None
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n📹 Video Info:")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Total Frames: {total_frames}")
    print(f"   Duration: {total_frames/fps:.2f} seconds")
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Create frames directory if saving frames
    if save_frames:
        os.makedirs(frames_dir, exist_ok=True)
    
    # Detection statistics
    stats = {
        'total_frames': total_frames,
        'processed_frames': 0,
        'total_detections': 0,
        'detections_per_class': {},
        'processing_time': 0
    }
    
    print(f"\n🔍 Starting detection with confidence threshold: {confidence_threshold}")
    start_time = time.time()
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Run YOLO detection
        results = yolo_model(frame, conf=confidence_threshold, verbose=False)
        
        # Get annotated frame
        annotated_frame = results[0].plot()
        
        # Count detections
        if results[0].boxes is not None:
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                class_name = yolo_model.names[class_id]
                
                stats['total_detections'] += 1
                stats['detections_per_class'][class_name] = \
                    stats['detections_per_class'].get(class_name, 0) + 1
        
        # Write frame
        out.write(annotated_frame)
        
        # Save individual frame if requested
        if save_frames:
            frame_path = os.path.join(frames_dir, f"frame_{frame_count:06d}.jpg")
            cv2.imwrite(frame_path, annotated_frame)
        
        frame_count += 1
        stats['processed_frames'] = frame_count
        
        # Show progress
        if show_progress and frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            clear_output(wait=True)
            print(f"⏳ Processing: {progress:.1f}% ({frame_count}/{total_frames} frames)")
    
    # Cleanup
    cap.release()
    out.release()
    
    stats['processing_time'] = time.time() - start_time
    
    clear_output(wait=True)
    print("✅ Detection Complete!")
    print(f"\n📊 Statistics:")
    print(f"   Processed Frames: {stats['processed_frames']}")
    print(f"   Total Detections: {stats['total_detections']}")
    print(f"   Processing Time: {stats['processing_time']:.2f} seconds")
    print(f"   Processing Speed: {stats['processed_frames']/stats['processing_time']:.2f} FPS")
    print(f"\n📦 Detections per Class:")
    for class_name, count in sorted(stats['detections_per_class'].items(), 
                                     key=lambda x: x[1], reverse=True):
        print(f"   - {class_name}: {count}")
    
    print(f"\n💾 Output saved to: {output_path}")
    
    return stats

# %% [markdown]
# ## 6. Alternative: Using Built-in YOLO Video Processing

# %%
def detect_with_yolo_builtin(video_path: str, output_dir: str = "runs"):
    """
    Use YOLO's built-in video processing
    Simpler but less customizable
    """
    print(f"🔍 Processing video: {video_path}")
    
    results = model(
        source=video_path,
        save=True,
        conf=0.25,
        project=output_dir,
        name="yolo12_detection"
    )
    
    print(f"✅ Results saved in: {output_dir}/yolo12_detection/")
    return results

# %% [markdown]
# ## 7. Run Detection

# %%
# === OPTION 1: Google Colab - Upload Video ===
# Uncomment the lines below if using Google Colab

# video_path = upload_video_colab()

# %%
# === OPTION 2: Local Jupyter - Enter Path ===
# Uncomment and modify the line below for local usage

# video_path = "path/to/your/video.mp4"

# %%
# === OPTION 3: Use Specified Video ===
# Use the provided video path

video_path = r"C:\Users\Ashch\Machine_Learning\Practice\Walkers Street.mp4"
if os.path.exists(video_path):
    print(f"✅ Video found: {video_path}")
else:
    print(f"❌ Video not found: {video_path}")
    video_path = None

# %%
# Run detection on the video
if video_path:
    stats = detect_objects_in_video(
        video_path=video_path,
        output_path="output_detected.mp4",
        confidence_threshold=0.25,
        show_progress=True,
        save_frames=False
    )

# %% [markdown]
# ## 8. Display Results

# %%
# Display the output video (works in Jupyter/Colab)
from IPython.display import Video

if os.path.exists("output_detected.mp4"):
    display(Video("output_detected.mp4", embed=True, width=800))

# %%
# Download the output video (Google Colab)
# Uncomment below to download

# from google.colab import files
# files.download("output_detected.mp4")

# %% [markdown]
# ## 9. Advanced: Interactive Widget Interface

# %%
# Interactive interface with widgets
class VideoDetector:
    def __init__(self):
        self.video_path = None
        self.setup_widgets()
    
    def setup_widgets(self):
        # Model selector
        self.model_dropdown = widgets.Dropdown(
            options=['yolo12n', 'yolo12s', 'yolo12m', 'yolo12l', 'yolo12x'],
            value='yolo12s',
            description='Model:'
        )
        
        # Confidence slider
        self.conf_slider = widgets.FloatSlider(
            value=0.25,
            min=0.05,
            max=0.95,
            step=0.05,
            description='Confidence:'
        )
        
        # File upload
        self.upload = widgets.FileUpload(accept='.mp4,.avi,.mov,.mkv,.wmv', multiple=False)
        self.upload.observe(self.on_upload, names='value')
        
        # Process button
        self.process_btn = widgets.Button(
            description='Start Detection',
            button_style='success'
        )
        self.process_btn.on_click(self.on_process)
        
        # Output area
        self.output = widgets.Output()
        
        # Display interface
        display(widgets.VBox([
            widgets.HTML("<h3>🎥 YOLOv12 Video Object Detection</h3>"),
            widgets.HBox([self.model_dropdown, self.conf_slider]),
            widgets.HBox([self.upload, self.process_btn]),
            self.output
        ]))
    
    def on_upload(self, change):
        with self.output:
            clear_output()
            if self.upload.value:
                filename, file_info = list(self.upload.value.items())[0]
                with open(filename, 'wb') as f:
                    f.write(file_info['content'])
                self.video_path = filename
                print(f"✅ Video uploaded: {filename}")
            else:
                print("No file uploaded")
    
    def on_process(self, btn):
        with self.output:
            if self.video_path is None:
                print("❌ Please upload a video first!")
                return
            
            clear_output()
            
            # Load selected model
            print(f"Loading {self.model_dropdown.value}...")
            detector_model = YOLO(f"{self.model_dropdown.value}.pt")
            
            # Process video
            detect_objects_in_video(
                video_path=self.video_path,
                model=detector_model,
                output_path="output_detected.mp4",
                confidence_threshold=self.conf_slider.value
            )

# Uncomment to use interactive interface
# detector = VideoDetector()

# %% [markdown]
# ## 10. Batch Processing Multiple Videos

# %%
def process_multiple_videos(video_folder: str, output_folder: str = "outputs"):
    """Process all videos in a folder"""
    
    os.makedirs(output_folder, exist_ok=True)
    
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv']
    video_files = [f for f in os.listdir(video_folder) 
                   if Path(f).suffix.lower() in video_extensions]
    
    print(f"📁 Found {len(video_files)} videos in {video_folder}")
    
    all_stats = {}
    
    for i, video_file in enumerate(video_files):
        print(f"\n{'='*50}")
        print(f"Processing video {i+1}/{len(video_files)}: {video_file}")
        print('='*50)
        
        input_path = os.path.join(video_folder, video_file)
        output_path = os.path.join(output_folder, f"detected_{video_file}")
        
        stats = detect_objects_in_video(
            video_path=input_path,
            output_path=output_path
        )
        
        all_stats[video_file] = stats
    
    return all_stats

# Example usage:
# stats = process_multiple_videos("my_videos_folder", "detected_outputs")

# %% [markdown]
# ## 11. Extract Specific Object Clips

# %%
def extract_clips_with_object(
    video_path: str,
    target_class: str,
    output_dir: str = "clips",
    min_clip_duration: float = 1.0
):
    """
    Extract video clips containing a specific object
    """
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    clip_frames = []
    clip_count = 0
    in_clip = False
    
    print(f"🔍 Searching for '{target_class}' in video...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        results = model(frame, verbose=False)
        
        # Check if target class is detected
        target_found = False
        if results[0].boxes is not None:
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                if model.names[class_id].lower() == target_class.lower():
                    target_found = True
                    break
        
        if target_found:
            annotated = results[0].plot()
            clip_frames.append(annotated)
            in_clip = True
        elif in_clip:
            # Save clip if long enough
            if len(clip_frames) >= min_clip_duration * fps:
                clip_path = os.path.join(output_dir, f"{target_class}_clip_{clip_count}.mp4")
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out = cv2.VideoWriter(clip_path, fourcc, fps, (width, height))
                for cf in clip_frames:
                    out.write(cf)
                out.release()
                print(f"💾 Saved clip: {clip_path} ({len(clip_frames)/fps:.1f}s)")
                clip_count += 1
            
            clip_frames = []
            in_clip = False
    
    cap.release()
    print(f"\n✅ Extracted {clip_count} clips containing '{target_class}'")

# Example usage:
# extract_clips_with_object("sample_video.mp4", "car", "car_clips")

# %% [markdown]
# ## Summary
# 
# This notebook provides:
# 1. **Basic Detection**: Process videos and annotate detected objects
# 2. **Interactive Interface**: Widget-based UI for easy use
# 3. **Batch Processing**: Process multiple videos at once
# 4. **Clip Extraction**: Extract clips containing specific objects
# 
# ### Tips:
# - Use `yolo12n` for fastest processing
# - Use `yolo12x` for highest accuracy
# - Adjust confidence threshold based on your needs
# - Lower values = more detections (more false positives)
# - Higher values = fewer detections (more precise)